### __EVALUACIÓN CON LANGFUSE__ 















07/04/26 16:33

In [1]:
from __future__ import annotations
import os
import pprint
import logging
from pathlib import Path
import time
import json
import itertools
import pathlib
import langfuse
from tqdm import tqdm
import torch
import pandas as pd
import ragas
from ragas import evaluate
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
#from langchain_weaviate import WeaviateVectorStore
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langgraph.graph import START, END, StateGraph
from pydantic import BaseModel, Field, ValidationError
from langchain_ollama import OllamaLLM, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from ragas.llms.base import LangchainLLMWrapper, llm_factory
from ragas.run_config import RunConfig
from ragas.embeddings import LangchainEmbeddingsWrapper, embedding_factory
import weaviate
from langchain_weaviate import WeaviateVectorStore
from langchain.embeddings.base import Embeddings
from langchain_core.retrievers import BaseRetriever
from sentence_transformers import CrossEncoder, SentenceTransformer
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, TypedDict
from datetime import datetime, timezone
from langsmith import Client
from langsmith.evaluation import evaluate as ls_evaluate
from ragas import EvaluationDataset, SingleTurnSample, evaluate as ragas_evaluate
from ragas.metrics.collections import ContextRecall, Faithfulness, ContextPrecision, AnswerCorrectness


from datasets import Dataset

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
load_dotenv(override=True)

from IPython.display import display, HTML, Image
display(HTML("<style>.container { width:98;} </style>"))
device = "cuda" if torch.cuda.is_available() else "cpu"
# Para el aviso de Triton, pesos y otros transformers 
logging.getLogger("xformers").setLevel(logging.ERROR)


# -- Configurar Langfuse --
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

LANGFUSE_SECRET_KEY = os.getenv('LANGFUSE_SECRET_KEY')
LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY')
LANGFUSE_BASE_URL = os.getenv('LANGFUSE_BASE_URL', 'http://localhost:3000')

if LANGFUSE_SECRET_KEY and LANGFUSE_PUBLIC_KEY:
    os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
    os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
    os.environ["LANGFUSE_BASE_URL"] = LANGFUSE_BASE_URL

langfuse_client = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_BASE_URL,
)
langfuse_handler = CallbackHandler()

print(f"Langfuse configurado: {langfuse_client is not None} | host={LANGFUSE_BASE_URL}")

/home/raglinux/env_rag/lib/python3.12/site-packages/langsmith/evaluation/_runner.py:56: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  (ast.Str, ast.Constant) if hasattr(ast, "Str") else (ast.Constant,)


Langfuse configurado: True | host=http://localhost:4000




Sistema completo para documentar y comparar los resultados del RAG variando:
- **Prompt** (distintas plantillas)
- **LLM** 
- **Modelo de embedding** (GTE, E5-Instruct)
- **Reranker** (con/sin BAAI/bge-reranker-v2-m3)
- **Multi-query** (con/sin)
- **Tipo de búsqueda** (similarity vs hybrid search)
- **Temperatura** (0.3, 0.5, 0.7)

Métricas evaluadas con LLM como juez:
- Answer Correctness
- Faithfulness
- Context Recall
- Context Precision

In [15]:
import torch
import gc
import time

def safe_empty_cache():
    """Intenta vaciar la cache CUDA de forma segura y hace gc.collect().
    No devuelve nada; maneja excepciones internas para evitar crashes."""
    try:
        if torch.cuda.is_available():
            # intentamos sincronizar primero (opcional, puede ayudar a atrapar errores)
            try:
                torch.cuda.synchronize()
            except Exception:
                # si synchronize falla, lo ignoramos aquí; el siguiente empty_cache puede fallar también
                pass

            try:
                torch.cuda.empty_cache()
            except Exception as e:
                print(f"[WARN] safe_empty_cache: excepción al llamar torch.cuda.empty_cache(): {e}")
    except Exception as e_outer:
        # por si algo extraño pasa en la comprobación is_available()
        print(f"[WARN] safe_empty_cache: excepción inesperada: {e_outer}")
    finally:
        # recogida de basura en Python (siempre)
        try:
            gc.collect()
        except Exception:
            pass

# Uso correcto:
if torch.cuda.is_available():
    safe_empty_cache()
    # un pequeño retardo puede ayudar a que el driver libere estructuras visibles en nvidia-smi
    time.sleep(0.25)
    print(f'Memoria GPU limpiada')

/tmp/ipykernel_2649507/3090266382.py:27: ResourceWarning: unclosed <socket.socket fd=99, family=2, type=1, proto=6, laddr=('127.0.0.1', 39548), raddr=('127.0.0.1', 11434)>
  gc.collect()
Failed to detach context
Traceback (most recent call last):
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/langfuse/_client/propagation.py", line 272, in _propagate_attributes
    yield
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 155, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/opentelemetry/context/contextvars_context.py", line 53, in detach
    self._current_context.reset(token)
ValueError: <Token var=<ContextVar name='current_context' default={} at 0x74bc7615d670> at 0x74b818330500> was created in a different Context


Memoria GPU limpiada


### Dataset de evaluación

Preguntas con respuestas de referencia (ground truth) y contextos esperados, específicas del dominio arqueológico.

# DATASET DE EVALUACIÓN

EVAL_DATASET_NAME = "RAG-IDEArq-eval-v2"

questions = [
    'What are the main theoretical models of Neolithic expansion in Europe?',
    'Quais são as datas mais antigas da extração de sílex na península central?',
    '¿Cuáles son las cronologías de las manifestaciones funerarias del Mesolítico en las distintas regiones peninsulares?',
    'Principales yacimientos de la Segunda Edad del Hierro en la provincia de León.',
    'Periodización del Bronce Final en el Levante de la Península Ibérica, cronología de las fases y principales ejemplos de yacimientos asignados a las mismas.',
    'Yacimientos Calcolíticos de la Península Ibérica  en los que se han hallado objetos de marfil.',
    'Cronología y districubión espacial del poblamiento neolítico en la Meseta Sur.',
    #'Dataciones más antiguas para el megalitismo en la zona Sureste de la Península Ibérica.',
    'Excavaciones de urgencia de la Junta de Andalucía en la provincia de Almería publicadas en 2001.',
    'In what year did the Siret brothers excavate the La Bastida de Totana site?',
    'Yacimiento com a data de C14 mais antiga das Ilhas Baleares.',
    'Yacimientos neolíticos situados a menos de 150 km de Casa Montero',
    'Datación más antigua y más reciente de los yacimientos calcolíticos en el área de Valencina de la concepción (Sevilla)',
    'Dime si esta datación es la más reciente de los yacimientos calcolíticos en el área de Valencia de la concepción (Sevilla): Valencina, Cerro de la Cabeza, Ladera Sur, -1377 +- 23 ',
    'Fecha más antigua para un yacimiento funerario megalítico en la Península Ibérica.',
    'Dataciones más antiguas (i.e, más altas) de yacimientos paleolíticos para cada comunidad autónoma',
    '¿Estas dataciones son del Neolítico? Comunitat Valenciana, 33.900 ± 60, Galicia, 31.690 ± 50, Región de Murcia, 12.030 ± 0',
    '¿Cuál es el yacimiento calcolítico más alejado de la ciudad de Jaén en la provincia de Jaén?'
]

ground_truths = [
    """These can be divided into two main positions: the first, known as demic diffusion, emphasises the movement of Neolithic societies and, by extension, agricultural practices; the second, referred to in the literature as cultural diffusion, focuses on the importance of the transmission of the Neolithic package—technology (e.g., pottery), plants, and domesticated animals—as a trigger for the expansion of the Neolithic.""",
    """A única mina de sílex do Neolítico no centro de Espanha é a de Casamontero (Madrid). A série completa de datas de radiocarbono é apresentada na Fig. 3 e a sua distribuição espacial na Fig. 4 do artigo de Díaz del Río e Consuegra, 2015.
Infelizmente, a amostra de Sus sp. não tinha colagénio suficiente para ser datada. O teste X2 mostra que todas as datas, com a única exceção da Beta-232890, são estatisticamente idênticas. Recentemente, foi enviada outra matriz de anel para datação num fragmento de fémur de Ovis aries. O resultado, 6200+/ -40 BP (Beta-295152), é estatisticamente igual a dez das onze datas anteriores.
Assumindo a hipótese plausível de que elas datam diferentes eventos de mineração, há uma probabilidade de 65% de que todos os episódios de mineração tenham ocorrido entre 5327-5215 cal BC (1σ), um período de tempo de apenas cem anos. Portanto, estas datas de radiocarbono não permitem observar a evolução espácio-temporal da exploração mineira, mas indicam que o principal episódio de atividade da Casa Montero durou pouco mais de um século, quatro gerações. Esta interpretação não é apenas possível, é provável.""",
    """Modelo construido a partir de muestras individuales de radiocarbono de esqueletos mesolíticos encontrados en los cementerios ibéricos. La diferencia más significativa entre la región mediterránea y la región cantábrica y Portugal se observa en la aparición de cementerios durante el Mesolítico temprano. Además, al evaluar los datos de las tres zonas de la Península Ibérica, se pueden identificar los siguientes patrones cronológicos.
- En la región mediterránea, las fechas de El Collado muestran que los cementerios aparecieron alrededor de 9475-9300 cal BP. Este tipo de práctica funeraria continuará en otros yacimientos cercanos, como Casa Corona y Cingle del Mas Nou. A diferencia del cementerio de El Collado, que estuvo en uso durante unos 1100 años, según las fechas obtenidas, en Casa Corona y Cingle del Mas Nou, su periodo de uso es mucho más breve (8007-7583 cal BP). Además, Cingle del Mas Nou se diferencia de otros yacimientos funerarios mesolíticos de la Península Ibérica en que los restos de siete individuos (completos e incompletos) fueron depositados en una única estructura.
- En la fachada atlántica de Portugal, las primeras pruebas de cementerios en el estuario de Muge datan de 8409-8030 cal BP (en Cabeço de Arruda, por ejemplo). Estos están asociados a grandes concheros de más de 5 m de espesor, utilizados durante un largo periodo de tiempo. En el estuario del Sado, las fechas son ligeramente más recientes que en Muge, comenzando alrededor del 8200 cal BP (por ejemplo, en Amoreiras). Está claro que entre el 8160 y el 7970 cal BP, los grupos mesolíticos enterraban sistemáticamente a todos o algunos de sus muertos en cementerios.
- Por último, las fechas de los yacimientos funerarios del norte de la Península Ibérica (costa cantábrica) con dos o más individuos indican que los primeros enterramientos mesolíticos agrupados fueron un poco más recientes (entre 7981 y 6636 cal BP). En cualquier caso, cabe destacar que, a diferencia de las otras dos zonas, en la mayoría de los yacimientos solo se ha documentado un único individuo o grupos mucho más reducidos, como en Los Canes y La Braña.""",
    """Los principales yacimientos de la Segunda Edad del Hierro en la provincia de León son los siguientes:
- Castro de la Edad de Hierro de Valencia de Don Juan
- La Muela (al otro lado de la carretera de acceso a Valencia de Don Juan)
- Antigua ciudad astur-romana de Lancia.
- Regueras de Arriba o San Martín de Torres.
- En la zona cántabra, los castros laciniaegos de la Mesa en Rioscuro, que se delimita con una potente muralla de módulos, fechada seguramente en los siglos II y I a.C., y el castro de La Zamora, en sosas de Laciana.
- También en la zona cántabra en el municipio de Puebla de Lillo se han podido reconocer ciertas piezas metalíticas de la Segunda Edad del Hierro, recogidas en el antiguo castro de Castiltejón.
- El castro de Chano en la comarca de Fornela.
- Peña del Castro (La Ercina), ubicado a 2 km de la localidad de la Ercina
- El Castrelín de San Juan dse Paluezas
- La Corona del Castro en Borrenes
- La Peña del Hombre en el ayuntamiento de Priaranza
- Castro de Columbrianos
- Peña Piñera, en la Vega de Espinareda
- Por último, se ha señalado un conjunto de castros en altitudes considerables en las sierras del Teleno, la Valdería y el Bierzo, algunos ya conocidos en la bibliografía y en la Carta Arqueológica de León, con grandes amurallamientos, que se extienden entre afloramientos rocosos de materiales de la era Primaria; yacimientos como Portillo de Xandequín en Pozos, Peña Rayada en Cunas, Alto de San Vicente-Los Conventos en Morla de la Valdería, Yera de los Piornos-Peña del Tren en Torneros de la Valdería, Sierra del Pueblo en Torneros de la Valdería, El Pajarín-La Formosida en Boisán (Lucillo) y los bercianos localizados en Folgoso de la Ribera, Torre del Bierzo y Molinaseca.""",
    """Bronce tardío o reciente (c. 1550/1500-1300/1250 cal BC):
- Oropesa la Vella
- Torreló d'Onda
- Les Raboses
- Altet de Palau
- Cap Prim
- Mas del Corral
- Cabezo Redondo
- Peña de Sax
- El Negret
- Illeta del Banyets
- Tabayá

Bronce final I (c. 1300/1250-1000 cal BC):
- Costamar
- Oropesa la Vella
- El Castellet
- Torrelló de Boverot
- Pic dels Corbs III y IV
- Cova d'en Pardo
- Cova de la Pastora
- Cap Prim
- Peña de Sax
- El Negret
- Tabayá
- Botx-Grupitex

Bronce final II (1000-850 cal BC):
- Ereta del Castellar
- El Castellet
- Torrelló de Boverot
- Pic dels Corbs V
- Solana del Castell I
- Mola d'Agres
- Tabayá
- Caramoro
- Botx

Bronce final III (850-725 cal BC):
- Ereta del Castellar
- El Castellet
- Torrelló de Boverot
- Vinarragell
- La Vital
- Solana del Castell II
- Mola d'Agres
- Cova de la Sarsa
- Tabayá
- Peña Negra I
- Barranc del Botx
- Saladares Ia1/IA2

Hierro antiguo o fase Orientalizante (725-550 cal BC):
- Vinarragell
- El Molón
- Los Villares
- Solana del Castell III
- El Castellar
- El Puig
- Camara
- Tabayá
- Peña Negra II
- Casa Secà
- Saladares IA3""",
    """Calcolítico antiguo (pre-campaniforme [bell beaker]):
- Zambujal
- Vila Nova de São Pedro
- Leceiaa
- Praia das Maçãs
- Palmela
- Alcalar
- Perdigões
- Señorío de Guzmán
- La Pijotilla
- Valencina de la Concepción
- Gilena
- Los Millares

Calcolítico reciente (campaniforme [bell beaker]):
- Palmela
- Pedra do Ouro
- Verdelha dos Ruivos
- Vila Nova de São Pedro
- Perdigões
- Valencina de la Concepción
- Los Algarbes
- Cerro de la Virgen
- Camino de Yeseras
- La Pijotilla""",
    """Lo dividimos en dos áreas: poblamiento neolítico en el valle medio y alto del Tajo y poblamiento neolítico de La Mancha.
- Valle del Tajo: En la zona de la Sierra madrileña se localizan las cuevas la Cueva de La Ventana o la Cueva de la Higuera. Ambas son especialmente importantes para comprender los asentamientos en cueva, ya que disponen de dataciones radiocarbónicas asociadas a contextos de habitación. Ya en la provincia de Guadalajara destacan los yacimientos de la Cueva de la Hoz, Abrigo de Tordelrrábano  la Cueva de Jarama II, los enclaves de Sorbe II (Humanes de Mohernando), II y VII, la Cueva del Paso, el Abrigo de los Enebrales, Cueva del Reno y la Cueva del Destete.  Sin embargo, son los yacimientos de hoyos los que constituyen el principal modelo de asentamiento. Las excavaciones arqueológicas en extensión realizadas en los últimos años han puesto de manifiesto el predominio del hábitat en asentamientos al aire libre durante este periodo. Tan sólo se han documentado 8 enclaves neolíticos serranos de un total de 24 yacimientos cartografiados en la región. Los 16 yacimientos restantes se ubican en zonas bajas de los valles. A estas cifras hay que sumar tres nuevos yacimientos neolíticos: Soto del Henares, La Serna y Prado de Galápagos, recientemente publicados por C. Blasco et al.
(2016) que amplían el listado de asentamientos de hoyos ubicados en los fondos de
valle.
La concentración de yacimientos neolíticos es especialmente interesante en la zona sureste de la Comunidad de Madrid, en los tramos finales de los ríos Henares, Jarama y Manzanares. La densidad de yacimientos de hoyos neolíticos en las zonas bajas de estos cursos fluviales ha aportado desde hace años datos muy interesantes para conocer los patrones de asentamiento y los modelos de
ocupación territorial de las primeras comunidades neolíticas.
- Poblamiento neolítico de La Mancha: El modelo de asentamiento neolítico mejor conocido en La Mancha son las ocupaciones en cuevas y abrigos. En este sentido, los yacimientos neolíticos manchegos mejor conocidos son la Cueva del Niño y el Abrigo de Molino de Vadico, en Albacete, y el Abrigo de Verdelpino, en Cuenca, a los que ya nos referimos en el capítulo anterior. En estos yacimientos se han documentado, además, ocupaciones previas, por lo que se son especialmente interesantes para estudiar el momento de adopción de los modos de vida neolíticos.
En los últimos años se han realizado varias intervenciones arqueológicas en el contexto de las obras de construcción de grandes infraestructuras. Desde el punto de vista arqueológico, este tipo de intervenciones implican la prospección y, en su caso, excavación en zonas aleatorias afectadas por los proyectos constructivos, por lo que no se introducen sesgos relacionados con intereses científicos concretos. Por lo tanto, los hallazgos suponen una buena aproximación de la existencia o ausencia de distintos tipos de registro arqueológico. En este sentido, las obras de acceso al aeropuerto de Ciudad Real o los trabajos de la autopista que conecta Ocaña (Toledo) con La Roda (Ciudad Real), no aportaron hallazgos de asentamientos neolíticos.
Si bien es cierto que las investigaciones sobre esta etapa de la Prehistoria en La Mancha han sido escasas y son pocos los equipos de investigación que han desarrollado líneas de investigación orientadas al conocimiento de las primeras sociedades productoras en esa región, no es menos cierto que las intervenciones arqueológicas llevadas a cabo en el contexto de obras de infraestructura en Castilla-La Mancha apoyan la idea de un poblamiento neolítico caracterizado por su escasa densidad, especialmente si se compara con otras regiones como en valle medio y alto del Tajo.""",
    """Son el solar situado en la avenida Pablo Iglesias esquina A Rafaeka Jiménez, solar situado en la calle La central de Villaricos (cuevas de Almanzora) y en la calle Castillejo (Gador, Almería).""",
    """In 1886.""",
    """Cova de Moleta (Sóller), 80000 BP""",
    """Los yacimientos neolíticos situados a menos de 150 km de Casa Montero son los siguientes:
- La Atalaya, situado en la provincia de Ávila (Muñopepe). Coordenadas: longitud -4,817, latitud 40,636.
- Portillo de las Cortes, situado en la provincia de Guadalajara (Anguita). Coordenadas: longitud -2,407, latitud 41,056.
- El Cañaveral, situado en la provincia de Madrid (Madrid). Coordenadas: longitud -3,56, latitud 40,407.
- Pista de Motos, situado en la provincia de Madrid (Madrid). Coordenadas: longitud -3,665, latitud 40,339.
- O'Donnell, situado en la provincia de Madrid (Madrid). Coordenadas: longitud -3,658, latitud 0,419.
- Casa Montero, situado en la provincia de Madrid (Madrid). Coordenadas: longitud -3,5257, latitud 40,4049.
- Colector H05, situado en la provincia de Madrid (Madrid). Coordenadas: longitud -3,671, latitud 40,351.
- Cueva de la Higuera, situado en la provincia de Madrid (Patones). Coordenadas: longitud -3,509, latitud 40,854.
- Capanegra/Deseada, situado en la provincia de Madrid (Rivas-Vaciamadrid). Coordenadas: longitud -3,52, latitud 40,355.
- El Congosto**, situado en la provincia de Madrid (Rivas-Vaciamadrid). Coordenadas: longitud -3,556, latitud 40,329.
- Cueva de la Ventana, situado en la provincia de Madrid (Torrelaguna). Coordenadas: longitud -3,527, latitud 40,851.
- Cueva de la Vaquera, situado en la provincia de Segovia (Torreiglesias). Coordenadas: longitud -4,059, latitud 41,086.
- La Mina, situado en la provincia de Soria (Alcubilla de las Peñas). Coordenadas: longitud -2,55, latitud 41,25.
- La Mina, situado en la provincia de Soria (Alcubilla de las Peñas). Coordenadas: longitud -2,55, latitud 41,25.
- Túmulo de la Sima**, situado en la provincia de Soria (Miño de Medinaceli). Coordenadas: longitud -2,54, latitud 41,177.
- Peña de la Abuela**, situado en la provincia de Soria (Miño de Medinaceli). Coordenadas: longitud -2,514, latitud 41,157.
- La Tarayuela, situado en la provincia de Soria (Miño de Medinaceli). Coordenadas: longitud -2,503, latitud 41,167.
- La Revilla del Campo**, situado en la provincia de Soria (Miño de Medinaceli). Coordenadas: longitud -2,515, latitud 41,17.
- Carlos Álvarez, situado en la provincia de Soria (Miño de Medinaceli). Coordenadas: longitud -2,546, latitud 41,187.
- La Lámpara, situado en la provincia de Soria (Miño de Medinaceli). Coordenadas: longitud -2,515, latitud 41,156.
- El Castillejo, situado en la provincia de Toledo (Huecas). Coordenadas: longitud -4,2092, latitud 39,9914.
- El Tonto, situado en la provincia de Toledo (Mocejón). Coordenadas: longitud -3,931, latitud 39,944.
- La Paleta, situado en la provincia de Toledo (Numancia de la Sagra). Coordenadas: longitud -3,864, latitud 40,085.
- La Ocañuela, situado en la provincia de Toledo (Ocaña). Coordenadas: longitud -3,565, latitud 39,96.""",
    """Datación más antigua: Valencina, Instituto de Educación Secundaria, 4800 +- 100
Datación más reciente: Valencina, Cerro de la Cabeza, Ladera Sur, 175 +- 20""",
    """No, es errónea. La datación correcta es: Valencina, Cerro de la Cabeza, Ladera Sur, 175 +- 20""",
    """Trikuaizti 2 (Gipuzkoa), 12015 +- 145""",
    """Las dataciones son las siguientes:
- Andalucía, 51.914 ± 45
- Aragón, 25.330 ± 80
- Cantabria, 48.200 ± 80
- Castilla y León, 30.300 ± 25
- Castilla-La Mancha, 28.660 ± 40
- Cataluña, 38.640 ± 50
- Comunidad de Madrid, 30.280 ± 28
- Comunidad Foral de Navarra, 21.600 ± 30
- Comunitat Valenciana, 33.900 ± 60
- Extremadura, 61.219 ± 70
- Galicia, 31.690 ± 50
- Illes Balears, 24.220 ± 115
- La Rioja, 6.220 ± 100
- País Vasco, 34.350 ± 130
- Principado de Asturias, 16.700 ± 30
- Región de Murcia, 12.030 ± 0""",
    """No, estas dataciones corresponden al período Paleolítico.""",
    """El yacimiento calcolítico más alejado de la ciudad de Jaén en la provincia de Jaén es el yacimiento de Eras del Alcázar, a aproximadamente 50 km de la ciudad de Jaén."""
]


# 1. Crear o recuperar el dataset en Langfuse
try:
    # Intentamos crear el dataset
    langfuse_client.create_dataset(
        name=EVAL_DATASET_NAME,
        description="Dataset arqueológico para evaluación RAG (local)"
    )
    print(f"Dataset '{EVAL_DATASET_NAME}' creado en Langfuse.")
except Exception as e:
    # Si ya existe, Langfuse dará un error, simplemente lo ignoramos
    print(f"Dataset '{EVAL_DATASET_NAME}' ya existe o no se pudo crear. Procediendo a actualizar items.")

# 2. Subir los items (Pregunta + Respuesta de referencia)
# Langfuse usa 'input' para la pregunta y 'expected_output' para la respuesta ideal
for q, gt in zip(questions, ground_truths):
    langfuse_client.create_dataset_item(
        dataset_name=EVAL_DATASET_NAME,
        input=q,               # La pregunta
        expected_output=gt,    # El ground truth
        metadata={
            "source": "Arqueo-Manual-Eval",
            "language": "multilingual"
        }
    )

print(f"Sincronizados {len(questions)} items en el dataset de Langfuse.")



### Prompts a comparar


PROMPTS = {
    "prompt_zero_shot": 
        """Eres un asistente experto en arqueología, historia y sistemas de información geográfica. Responde la pregunta usando los contextos proporcionados, que pueden estar en español, inglés, francés, catalán o portugués.
Sintetiza información de todos los contextos relevantes independientemente de su idioma. Si encuentras información relevante en cualquier idioma, úsala para construir tu respuesta en el mismo idioma en el que se realiza la pregunta.

Contexto: {context}

Pregunta: {question}

Instrucciones:
- Si encuentras información parcial en el contexto, intégrala en la respuesta aunque no sea completa.
- Si no hay absolutamente nada relevante, responde claramente: "No hay información suficiente en el contexto".
- NO INVENTES NI ALUCINES INFORMACIÓN.
- Cuando sea posible, cita explícitamente los puntos clave del contexto (ej. autores, años, títulos de publicaciones, yacimientos, cronologías, dataciones, coordenadas).
- Haz cálculos utilizando la distancia euclidiana, después transforma los grados a kilómetros y proporciona los nombres de los yacimientos.
- Responde siempre de forma clara, estructurada y útil para un investigador.

Basándote solo en la información anterior, responde: 

Respuesta:"""
,

    "prompt_one_shot": 
        """Eres un asistente experto en arqueología, historia y sistemas de información geográfica. Responde la pregunta usando los contextos proporcionados, que pueden estar en español, inglés, francés, catalán o portugués.
Sintetiza información de todos los contextos relevantes independientemente de su idioma. Si encuentras información relevante en cualquier idioma, úsala para construir tu respuesta en el mismo idioma en el que se realiza la pregunta.

Contexto: {context}

Pregunta: {question}

Instrucciones:
- Si encuentras información parcial en el contexto, intégrala en la respuesta aunque no sea completa.
- Si no hay absolutamente nada relevante, responde claramente: "No hay información suficiente en el contexto".
- NO INVENTES NI ALUCINES INFORMACIÓN.
- Cuando sea posible, cita explícitamente los puntos clave del contexto (ej. autores, años, títulos de publicaciones, yacimientos, cronologías, dataciones, coordenadas).
- Haz cálculos utilizando la distancia euclidiana, después transforma los grados a kilómetros y proporciona los nombres de los yacimientos.
- Responde siempre de forma clara, estructurada y útil para un investigador.

Ejemplo:
Q: ¿Cuál es la utilidad de los análisis de isótopos de estroncio en Arqueología?
A: Es útil para inferir movilidad y origen geográfico de individuos mediante la comparación de firmas isotópicas. Es un método cuantitativo y comparable para estudiar desplazamientos en arqueología (Larsen 2018).

Basándote solo en la información anterior, responde: 

Respuesta:"""
,

    "prompt_few_shot":
        """Eres un asistente experto en arqueología, historia y sistemas de información geográfica. Responde la pregunta usando los contextos proporcionados, que pueden estar en español, inglés, francés, catalán o portugués.
Sintetiza información de todos los contextos relevantes independientemente de su idioma. Si encuentras información relevante en cualquier idioma, úsala para construir tu respuesta en el mismo idioma en el que se realiza la pregunta.
Contexto: {context}

Pregunta: {question}

Instrucciones:
- Si encuentras información parcial en el contexto, intégrala en la respuesta aunque no sea completa.
- Si no hay absolutamente nada relevante, responde claramente: "No hay información suficiente en el contexto".
- NO INVENTES NI ALUCINES INFORMACIÓN.
- Cuando sea posible, cita explícitamente los puntos clave del contexto (ej. autores, años, títulos de publicaciones, yacimientos, cronologías, dataciones, coordenadas).
- Haz cálculos utilizando la distancia euclidiana, después transforma los grados a kilómetros y proporciona los nombres de los yacimientos.
- Responde siempre de forma clara, estructurada y útil para un investigador.

Ejemplos:
Q: ¿Cuál es la utilidad de los análisis de isótopos de estroncio en Arqueología?
A: Permiten evaluar movilidad y procedencia comparando firmas isotópicas entre individuos y entornos. Es un enfoque cuantitativo para estudiar desplazamientos (Larsen 2018).

Q: ¿Cuáles son las características de la distribución geográfica de la muestra disponible de análisis de isótopos de estroncio en la Península Ibérica?
A: La muestra es desigual y discontinua. Hay concentraciones en focos concretos (Lisboa, valle del Ebro), cobertura geográfica y cronológica irregular, y escasez extrema en periodos como el Mesolítico. La excepción es la Edad del Cobre, con mayor volumen de datos.

Basándote solo en la información anterior, responde: 

Respuesta:"""
,
}

print(f"Prompts definidos: {list(PROMPTS.keys())}")

# Subir los prompts a Langfuse (Prompt Management)
for name, text_template in PROMPTS.items():
    try:
        langfuse_client.create_prompt(
            name=name,
            prompt=text_template,
            labels=["production"],
            type="text"

        )
        print(f"Prompt '{name}' subido con éxito.")
    except Exception as e:
        print(f"Error con '{name}': {e}")

In [ ]:
# Refuerzo de prompts: justificación obligatoria con evidencia del contexto
EVIDENCE_BLOCK = """
Reglas de justificación obligatoria:
- Cada afirmación factual importante debe incluir una línea de evidencia explícita tomada del contexto.
- Formato obligatorio de salida:
  1) Respuesta
  2) Evidencias (lista con al menos 2 evidencias cuando sea posible)
  3) Limitaciones
- Si no hay evidencia suficiente, responde exactamente: "No hay información suficiente en el contexto".
- No uses conocimiento externo para completar huecos.
""".strip()

for key in list(PROMPTS.keys()):
    if "Reglas de justificación obligatoria" not in PROMPTS[key]:
        PROMPTS[key] = PROMPTS[key].replace("Respuesta:", f"{EVIDENCE_BLOCK}\n\nRespuesta:")

# Publicar variante de evaluación con etiqueta específica
for name, text_template in PROMPTS.items():
    try:
        langfuse_client.create_prompt(
            name=f"{name}_eval_v2",
            prompt=text_template,
            labels=["production", "eval-v2", "evidence-required"],
            type="text",
        )
        print(f"[OK] Prompt '{name}_eval_v2' actualizado.")
    except Exception as e:
        print(f"[WARN] Prompt '{name}_eval_v2': {e}")

### Construcción del RAG

Ejecuta todas las preguntas del dataset y registra los resultados en Langfuse.

In [6]:
from triton import Config

class E5InstructEmbeddingsEval(Embeddings):
    def __init__(self, model_name="intfloat/multilingual-e5-large-instruct", device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device
        self.model = SentenceTransformer(model_name, device=device)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        prefixed = [f"passage: {t}" for t in texts]
        return self.model.encode(prefixed, device=self.device, show_progress_bar=False).tolist()

    def embed_query(self, text: str) -> List[float]:
        return self.model.encode([f"query: {text}"], device=self.device, show_progress_bar=False)[0].tolist()


# CONFIGURACIONES DE EXPERIMENTOS

# Modelos de embedding disponibles (se instancian bajo demanda)
EMBEDDING_CONFIGS = {
    "e5-large-instruct": {
        "class": E5InstructEmbeddingsEval,
        "kwargs": {"model_name": "intfloat/multilingual-e5-large-instruct"},
        "weaviate_index": "BgeE5Chunk1000Sim05",
    }
}

# LLMs disponibles (Ollama)
LLM_CONFIGS = {
    "mistral-small-24b-instruct": {"model": "hf.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF:Q5_K_M", "num_ctx": 16384},
    "qwen2.5-14b": {"model": "qwen2.5:14b", "num_ctx": 16384},
}


### Consulta GeoJSON directa

In [34]:
import math
import re
import unicodedata
from pathlib import Path
from typing import List, Any, Optional

import geopandas as gpd
from shapely.validation import make_valid

GEOJSON_PATH = Path("capas/c14_v2.geojson")


def _nl_normalize(text: str) -> str:
    text = text or ""
    text = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in text if unicodedata.category(ch) != "Mn").lower().strip()


def _clean_html_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value).replace("<br/>", " | ").replace("<br>", " | ").strip()


def _safe_text(row: pd.Series, key: str, default: str = "") -> str:
    val = row.get(key, default)
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return default
    return str(val).strip()


def _extract_lon_lat(geom) -> tuple[Optional[float], Optional[float]]:
    if geom is None or geom.is_empty or not geom.is_valid:
        return (None, None)

    if geom.geom_type == "Point":
        return (float(geom.x), float(geom.y))

    try:
        rp = geom.representative_point()
        return (float(rp.x), float(rp.y))
    except Exception:
        return (None, None)


def _euclidean_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    # Distancia euclidiana en grados y conversión aproximada a km.
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    d_deg = math.sqrt((dlat ** 2) + (dlon ** 2))
    return d_deg * 111.32


def _is_distance_question(question: str) -> bool:
    qn = _nl_normalize(question)
    keys = [
        "distancia", "distance", "km", "kilomet",
        "menos de", "mas cercano", "mas alejado", "nearest", "farthest"
    ]
    return any(k in qn for k in keys)


def _infer_anchor_site(question: str, geojson_data: gpd.GeoDataFrame) -> Optional[pd.Series]:
    qn = _nl_normalize(question)
    q_tokens = [t for t in re.findall(r"[a-z]{4,}", qn)]
    if not q_tokens:
        return None

    best_idx = None
    best_score = 0
    for idx, row in geojson_data.iterrows():
        name_norm = _nl_normalize(row.get("_nombre", ""))
        if not name_norm:
            continue
        score = 0
        if name_norm in qn:
            score += 100
        for tok in q_tokens:
            if tok in name_norm:
                score += 1
        if score > best_score:
            best_score = score
            best_idx = idx

    if best_idx is None or best_score < 2:
        return None
    return geojson_data.loc[best_idx]


def _distance_geo_docs(question: str, geojson_data: gpd.GeoDataFrame, top_k: int = 10) -> List[Document]:
    if geojson_data is None or geojson_data.empty or not _is_distance_question(question):
        return []

    anchor = _infer_anchor_site(question, geojson_data)
    if anchor is None:
        return []

    anchor_lon, anchor_lat = _extract_lon_lat(anchor.geometry)
    if anchor_lat is None or anchor_lon is None:
        return []

    rows = []
    for idx, row in geojson_data.iterrows():
        lon, lat = _extract_lon_lat(row.geometry)
        if lat is None or lon is None:
            continue
        d_km = _euclidean_km(anchor_lat, anchor_lon, lat, lon)
        rows.append((d_km, idx, row, lon, lat))

    if not rows:
        return []

    qn = _nl_normalize(question)
    wants_farthest = any(k in qn for k in ["mas alejado", "mas lejano", "farthest"])
    rows.sort(key=lambda x: x[0], reverse=wants_farthest)

    anchor_name = anchor.get("_nombre", "Referencia")
    docs = []
    for d_km, idx, row, lon, lat in rows[:top_k]:
        name = row.get("_nombre", "Desconocido") or "Desconocido"
        territory = row.get("_territorio", "N/A") or "N/A"
        content = (
            f"DISTANCIA_GEO: Distancia entre '{anchor_name}' y '{name}': {d_km:.2f} km (Euclidiana).\n"
            f"REFERENCIA: {anchor_name} ({anchor_lat:.6f}, {anchor_lon:.6f})\n"
            f"OBJETIVO: {name} ({lat:.6f}, {lon:.6f})\n"
            f"TERRITORIO: {territory}\n"
            f"---"
        )
        docs.append(
            Document(
                page_content=content,
                metadata={
                    "source": str(GEOJSON_PATH),
                    "feature_index": str(idx),
                    "distance_km": float(d_km),
                    "anchor_site": str(anchor_name),
                    "yacimiento": str(name),
                },
            )
        )

    return docs


def load_geojson(path: Path) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    if gdf.empty:
        raise ValueError(f"GeoJSON sin features: {path}")
    if "geometry" not in gdf.columns:
        raise ValueError("GeoJSON sin columna geometry")

    if gdf.crs is None:
        print("[WARN] GeoJSON sin CRS explícito. Se asume EPSG:4326.")
        gdf = gdf.set_crs(epsg=4326, allow_override=True)

    # Normalizamos a EPSG:4326 para coherencia geográfica en metadatos.
    if gdf.crs and gdf.crs.to_epsg() != 4326:
        print(f"[INFO] CRS detectado: {gdf.crs}. Reproyectando a EPSG:4326.")
        gdf = gdf.to_crs(epsg=4326)
    else:
        print(f"[INFO] CRS detectado: {gdf.crs}")

    gdf = gdf.copy()

    # Reparación de geometrías inválidas de forma defensiva.
    invalid_before = int((~gdf.geometry.is_valid).sum())
    if invalid_before > 0:
        gdf["geometry"] = gdf.geometry.apply(lambda geom: make_valid(geom) if geom is not None and not geom.is_valid else geom)
        invalid_after = int((~gdf.geometry.is_valid).sum())
        print(f"[WARN] Geometrías inválidas reparadas: {invalid_before - invalid_after} | restantes: {invalid_after}")

    # Campos requeridos para retrieval robusto.
    for col in ["yacimiento_id", "yacimiento", "unidad_territorial", "tipologia_crono", "dataciones_c_14"]:
        if col not in gdf.columns:
            gdf[col] = ""

    gdf["_nombre"] = gdf["yacimiento"].apply(lambda v: _safe_text(pd.Series({"v": v}), "v", "Desconocido"))
    gdf["_territorio"] = gdf["unidad_territorial"].apply(lambda v: _safe_text(pd.Series({"v": v}), "v", ""))
    gdf["_tipologia"] = gdf["tipologia_crono"].apply(_clean_html_text)
    gdf["_dataciones"] = gdf["dataciones_c_14"].apply(_clean_html_text)
    gdf["_search_blob"] = (
        gdf["_nombre"].apply(_nl_normalize)
        + " "
        + gdf["_territorio"].apply(_nl_normalize)
        + " "
        + gdf["_tipologia"].apply(_nl_normalize)
        + " "
        + gdf["_dataciones"].apply(_nl_normalize)
    )

    return gdf


def query_geojson(question: str, geojson_data: gpd.GeoDataFrame, top_k: int = 10) -> List[Document]:
    qn = _nl_normalize(question)
    tokens = [t for t in re.findall(r"[a-záéíóúñçü]{4,}", qn)]

    if geojson_data is None or geojson_data.empty:
        return []

    scored = []
    for idx, row in geojson_data.iterrows():
        search_blob = row.get("_search_blob", "")
        if not search_blob:
            continue

        score = 0
        for tok in tokens:
            if tok in _nl_normalize(row.get("_nombre", "")):
                score += 8
            elif tok in _nl_normalize(row.get("_territorio", "")):
                score += 6
            elif tok in _nl_normalize(row.get("_tipologia", "")):
                score += 4
            elif tok in search_blob:
                score += 2

        if score <= 0:
            continue

        lon, lat = _extract_lon_lat(row.geometry)
        coord_text = "sin coordenadas válidas"
        if lat is not None and lon is not None:
            coord_text = f"Latitud {lat:.6f}, Longitud {lon:.6f}"

        nombre = row.get("_nombre", "Desconocido") or "Desconocido"
        territorio = row.get("_territorio", "") or "N/A"
        tipologia = row.get("_tipologia", "") or "N/A"
        dataciones = row.get("_dataciones", "") or "N/A"

        content = (
            f"YACIMIENTO: {nombre}\n"
            f"UBICACIÓN TERRITORIAL: {territorio}\n"
            f"DATOS GEOGRÁFICOS: {coord_text}\n"
            f"CRONOLOGÍA: {tipologia}\n"
            f"DATACIONES C14: {dataciones}\n"
            f"---"
        )

        scored.append(
            {
                "score": score,
                "doc": Document(
                    page_content=content,
                    metadata={
                        "id": row.get("yacimiento_id"),
                        "yacimiento": nombre,
                        "territorio": territorio,
                        "longitude": lon,
                        "latitude": lat,
                        "geometry_type": getattr(row.geometry, "geom_type", None),
                        "source": str(GEOJSON_PATH),
                        "feature_index": str(idx),
                    },
                ),
            }
        )

    scored.sort(key=lambda x: x["score"], reverse=True)
    lexical_docs = [s["doc"] for s in scored[:top_k]]
    dist_docs = _distance_geo_docs(question, geojson_data, top_k=top_k)

    # Dedupe por contenido conservando prioridad de docs de distancia cuando aplica.
    seen = set()
    merged = []
    for d in (dist_docs + lexical_docs):
        key = d.page_content[:300]
        if key in seen:
            continue
        seen.add(key)
        merged.append(d)
    return merged[:top_k]

In [50]:
# Validación rápida de lectura GeoJSON y recuperación por keywords
geo_preview = load_geojson(GEOJSON_PATH)
print(f"[OK] Features cargadas: {len(geo_preview)}")

geo_docs_preview = query_geojson("yacimientos calcoliticos en jaen", geo_preview, top_k=3)
print(f"[OK] Resultados geo recuperados: {len(geo_docs_preview)}")
if geo_docs_preview:
    print(geo_docs_preview[0].page_content[:300])

[INFO] CRS detectado: EPSG:4258. Reproyectando a EPSG:4326.
[OK] Features cargadas: 4478
[OK] Resultados geo recuperados: 3
YACIMIENTO: Grañena Baja
UBICACIÓN TERRITORIAL: Jaén (Jaén, ES)
DATOS GEOGRÁFICOS: Latitud 37.872946, Longitud -3.773173
CRONOLOGÍA: Otros, Yacimiento negativo/Campo de hoyos (Neolítico)
DATACIONES C14: CNA2897 5614±36 B.P. sobre Hueso (AMS) | Beta459523 5610±30 B.P. sobre Hueso (AMS)
---


In [8]:
from langchain_core.outputs import LLMResult

class JSONCleaningLLM(LangchainLLMWrapper):
    """Obliga al juez local a entregar JSON válido para RAGAS."""
    def generate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        res = super().generate_text(prompt, n, temperature, stop, callbacks)
        # Limpiamos Markdown o texto extra que Ollama suele añadir
        text = res.generations[0][0].text
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        elif "{" in text:
            text = "{" + text.split("{", 1)[1].rsplit("}", 1)[0] + "}"
        res.generations[0][0].text = text.strip()
        return res

In [30]:
def build_rag_pipeline(
    embedding,
    llm,
    prompt,
    weaviate_client,
    geojson_data,
    temperature,
    k,
    use_multiquery,
    use_reranker,
    use_hybrid=False,
    hybrid_alpha=0.5,
    k_multi=20,
    reranker_top_n=5,
    **kwargs,
):
    from langchain_core.prompts import PromptTemplate

    # 1. Embeddings con manejo de timeout
    cfg_emb = EMBEDDING_CONFIGS[embedding]
    emb_model = cfg_emb["class"](**cfg_emb["kwargs"])

    # 2. VectorStore
    vs = WeaviateVectorStore(
        client=weaviate_client,
        embedding=emb_model,
        index_name=cfg_emb["weaviate_index"],
        text_key="page_content"
    )
    collection = weaviate_client.collections.get(cfg_emb["weaviate_index"])

    k = int(k)
    k_multi = int(k_multi)
    reranker_top_n = int(reranker_top_n)

    # Recuperación inicial amplia para luego comprimir/reordenar.
    if use_multiquery:
        initial_k = max(k_multi, reranker_top_n)
    elif use_reranker:
        initial_k = max(k, reranker_top_n)
    else:
        initial_k = k

    class NativeHybridRetriever(BaseRetriever):
        collection: Any
        embedder: Any
        alpha: float = 0.5
        limit: int = 10
        fallback_retriever: Any = None

        @staticmethod
        def _obj_to_document(obj) -> Document:
            props = dict(getattr(obj, "properties", {}) or {})
            text = props.get("page_content") or props.get("text")
            if not text:
                text_parts = [str(v) for v in props.values() if isinstance(v, (str, int, float))]
                text = " | ".join(text_parts)

            meta = {"source": "weaviate_hybrid", "uuid": str(getattr(obj, "uuid", ""))}
            obj_meta = getattr(obj, "metadata", None)
            if obj_meta is not None:
                score = getattr(obj_meta, "score", None)
                distance = getattr(obj_meta, "distance", None)
                if score is not None:
                    try:
                        meta["hybrid_score"] = float(score)
                    except Exception:
                        meta["hybrid_score"] = score
                if distance is not None:
                    try:
                        meta["vector_distance"] = float(distance)
                    except Exception:
                        meta["vector_distance"] = distance
            return Document(page_content=str(text or ""), metadata=meta)

        def _get_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
            try:
                query_vector = self.embedder.embed_query(query)
                try:
                    res = self.collection.query.hybrid(
                        query=query,
                        vector=query_vector,
                        alpha=float(self.alpha),
                        limit=int(self.limit),
                    )
                except TypeError:
                    res = self.collection.query.hybrid(
                        query=query,
                        vector=query_vector,
                        alpha=float(self.alpha),
                        limit=int(self.limit),
                        return_metadata=["score", "distance"],
                    )

                objects = list(getattr(res, "objects", []) or [])
                docs = [self._obj_to_document(o) for o in objects]
                if docs:
                    return docs
            except Exception as e:
                print(f"[WARN] Hybrid nativo falló ({e}).")

            if self.fallback_retriever is not None:
                return self.fallback_retriever.invoke(query)
            return []

    search_kwargs = {"k": initial_k}
    similarity_retriever = vs.as_retriever(search_type="similarity", search_kwargs=search_kwargs)
    if use_hybrid:
        base_retriever = NativeHybridRetriever(
            collection=collection,
            embedder=emb_model,
            alpha=float(hybrid_alpha),
            limit=int(initial_k),
            fallback_retriever=similarity_retriever,
        )
        retrieval_mode = "hybrid_native"
    else:
        base_retriever = similarity_retriever
        retrieval_mode = "similarity"

    # 3. Multi-Query controlado: 3-5 reformulaciones con sinónimos/traducciones.
    retriever = base_retriever
    if use_multiquery:
        mq_llm = OllamaLLM(model=LLM_CONFIGS[llm]["model"], temperature=0.1)
        mq_prompt = PromptTemplate(
            input_variables=["question"],
            template=(
                "Eres un asistente de reformulación para recuperación semántica multilingüe.\n"
                "Genera exactamente 4 variantes de la consulta original.\n"
                "Reglas:\n"
                "- Mantén la intención exacta de la pregunta.\n"
                "- Incluye sinónimos técnicos relevantes de arqueología/SIG.\n"
                "- Incluye al menos 1 variante traducida (ES/EN/PT según proceda).\n"
                "- Una variante debe enfatizar cronología/tiempo y otra geografía/lugar cuando aplique.\n"
                "- Devuelve solo 4 líneas, una por variante, sin numeración.\n"
                "Consulta original: {question}"
            ),
        )
        retriever = MultiQueryRetriever.from_llm(
            retriever=base_retriever,
            llm=mq_llm,
            prompt=mq_prompt,
            include_original=True,
        )

    # 4. Reranker
    if use_reranker:
        reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
        compressor = CrossEncoderReranker(model=reranker_model, top_n=reranker_top_n)
        retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)

    # 5. Obtener Prompt de Langfuse
    lf_prompt = langfuse_client.get_prompt(prompt)
    prompt_tpl = ChatPromptTemplate.from_template(lf_prompt.prompt)

    # 6. LLM generador
    llm_obj = OllamaLLM(**LLM_CONFIGS[llm], temperature=temperature)

    def combined_retrieval(q: str) -> str:
        w_docs = retriever.invoke(q)
        g_docs = query_geojson(q, geojson_data, top_k=max(5, k))
        return "\n\n".join([d.page_content for d in (w_docs + g_docs)])

    chain = ({"context": combined_retrieval, "question": RunnablePassthrough()} | prompt_tpl | llm_obj | StrOutputParser())
    return chain, retriever, retrieval_mode

### Evaluadores RAGAS 

Define los evaluadores que calculan las 4 métricas: **Answer Correctness**, **Faithfulness**, **Context Recall** y **Context Precision**.

In [17]:
from ragas import EvaluationDataset, SingleTurnSample, evaluate as ragas_evaluate
from ragas.metrics import Faithfulness, AnswerCorrectness, ContextPrecision, ContextRecall                  
import uuid

# Juez local (usar un modelo válido en Ollama)
RAGAS_JUDGE_MODEL = os.getenv("RAGAS_JUDGE_MODEL", "qwen2.5:14b")
judge_llm = ChatOllama(model=RAGAS_JUDGE_MODEL, temperature=0.1, format="json", num_ctx=32768)
ragas_llm = LangchainLLMWrapper(judge_llm) 
ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
ragas_metrics = [Faithfulness(llm=ragas_llm), AnswerCorrectness(llm=ragas_llm, embeddings=ragas_emb), ContextPrecision(llm=ragas_llm), ContextRecall(llm=ragas_llm)]

# Configuración de ejecución
ragas_run_config = RunConfig(max_workers=1, timeout=900)

_GARBAGE_PLACEHOLDER = "No se encontró información relevante."

def _is_garbage_response(text: str) -> bool:
    if not text or len(str(text).strip()) < 10: return True
    t = str(text).strip()
    if '<|' in t or '|>' in t or 'assistant' in t.lower()[:5]: return True
    letters = sum(1 for c in t if c.isalpha())
    if len(t) > 20 and (letters / len(t)) < 0.3: return True
    return False


/tmp/ipykernel_1310907/2386388208.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerCorrectness, ContextPrecision, ContextRecall
/tmp/ipykernel_1310907/2386388208.py:2: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Faithfulness, AnswerCorrectness, ContextPrecision, ContextRecall
/tmp/ipykernel_1310907/2386388208.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Fai

In [26]:
import hashlib

METRIC_NAMES = ["faithfulness", "answer_correctness", "context_recall", "context_precision"]
DEFAULT_SCORE_WEIGHTS = {
    "faithfulness": 0.25,
    "answer_correctness": 0.25,
    "context_recall": 0.25,
    "context_precision": 0.25,
}


def _config_hash(config: dict) -> str:
    stable = json.dumps(config, sort_keys=True, ensure_ascii=False)
    return hashlib.md5(stable.encode("utf-8")).hexdigest()[:12]


def _normalize_score_weights(weights: Optional[dict] = None) -> dict:
    merged = dict(DEFAULT_SCORE_WEIGHTS)
    if weights:
        for k, v in weights.items():
            if k in merged:
                merged[k] = float(v)
    total = sum(max(0.0, float(v)) for v in merged.values())
    if total <= 0:
        return dict(DEFAULT_SCORE_WEIGHTS)
    return {k: max(0.0, float(v)) / total for k, v in merged.items()}


def _compute_global_score_from_means(row: dict, weights: Optional[dict] = None) -> Optional[float]:
    norm_w = _normalize_score_weights(weights)
    score_sum = 0.0
    weight_sum = 0.0
    for metric in METRIC_NAMES:
        value = row.get(f"{metric}_mean")
        if value is None or pd.isna(value):
            continue
        w = norm_w.get(metric, 0.0)
        score_sum += float(value) * w
        weight_sum += w
    if weight_sum <= 0:
        return None
    return float(score_sum / weight_sum)


def _build_run_metadata(config: dict, retrieval_mode: str, exec_mode: str) -> dict:
    metadata = {
        "embedding_model": config.get("embedding"),
        "generator_model": config.get("llm"),
        "retrieval_mode": retrieval_mode,
        "use_hybrid": bool(config.get("use_hybrid", False)),
        "hybrid_alpha": config.get("hybrid_alpha") if bool(config.get("use_hybrid", False)) else None,
        "use_multiquery": bool(config.get("use_multiquery", False)),
        "use_reranker": bool(config.get("use_reranker", False)),
        "temperature": config.get("temperature"),
        "execution_mode": exec_mode,
        "config_hash": _config_hash(config),
        "experiment_ts_utc": datetime.now(timezone.utc).isoformat(),
    }
    metadata.update({k: str(v) for k, v in config.items()})
    return metadata


def _preflight_or_raise(w_client, collection_name: str, dataset_name: str, generator_model: str):
    if not LANGFUSE_SECRET_KEY or not LANGFUSE_PUBLIC_KEY:
        raise RuntimeError("Faltan LANGFUSE_SECRET_KEY o LANGFUSE_PUBLIC_KEY en el entorno.")

    try:
        _ = langfuse_client.get_dataset(dataset_name)
    except Exception as e:
        raise RuntimeError(f"No se pudo acceder a Langfuse o al dataset '{dataset_name}': {e}")

    if not w_client.is_ready():
        raise RuntimeError("Weaviate no está listo. Verifica contenedor y puertos 8080/50051.")

    try:
        w_client.collections.get(collection_name)
    except Exception as e:
        raise RuntimeError(f"No existe la colección '{collection_name}' en Weaviate: {e}")

    if not GEOJSON_PATH.exists():
        raise RuntimeError(f"No se encontró el GeoJSON requerido: {GEOJSON_PATH}")

    try:
        llm_probe = OllamaLLM(model=LLM_CONFIGS[generator_model]["model"], temperature=0.0)
        _ = llm_probe.invoke("Responde solo 'ok'.")
    except Exception as e:
        raise RuntimeError(f"No se pudo invocar el generador '{generator_model}' en Ollama: {e}")


def _log_metric_to_langfuse(trace_id: str, observation_id: Optional[str], metric_name: str, metric_value: float) -> bool:
    try:
        langfuse_client.create_score(
            trace_id=trace_id,
            observation_id=observation_id,
            name=f"ragas_{metric_name}",
            value=float(metric_value),
            data_type="NUMERIC",
            comment="RAGAS evaluation metric",
        )
        return True
    except Exception as e:
        try:
            langfuse_client.create_event(
                trace_context={"trace_id": trace_id},
                name=f"ragas_{metric_name}_fallback",
                output={"value": float(metric_value)},
                metadata={"error": str(e), "source": "ragas_fallback"},
                level="WARNING",
            )
            return True
        except Exception as inner_e:
            print(f"[WARN] No se pudo enviar métrica {metric_name} al trace {trace_id}: {inner_e}")
            return False


def _log_aggregates_to_langfuse(
    run_name: str,
    config: dict,
    retrieval_mode: str,
    exec_mode: str,
    df_results: pd.DataFrame,
    score_weights: Optional[dict] = None,
):
    agg_trace_id = langfuse_client.create_trace_id(seed=f"agg-{run_name}")
    run_metadata = _build_run_metadata(config, retrieval_mode, exec_mode)

    for metric in METRIC_NAMES:
        if metric not in df_results.columns:
            continue
        series = df_results[metric].dropna()
        if series.empty:
            continue

        stats = {
            "mean": float(series.mean()),
            "median": float(series.median()),
            "std": float(series.std(ddof=0)),
            "min": float(series.min()),
            "max": float(series.max()),
            "p25": float(series.quantile(0.25)),
            "p75": float(series.quantile(0.75)),
            "n_items": float(len(series)),
        }

        for stat_name, stat_value in stats.items():
            langfuse_client.create_score(
                trace_id=agg_trace_id,
                name=f"agg_{metric}_{stat_name}",
                value=stat_value,
                data_type="NUMERIC",
                comment=f"Aggregate {stat_name} for {metric}",
            )

    mean_row = {}
    for metric in METRIC_NAMES:
        series = df_results[metric].dropna() if metric in df_results.columns else pd.Series(dtype=float)
        mean_row[f"{metric}_mean"] = float(series.mean()) if not series.empty else None
    global_score = _compute_global_score_from_means(mean_row, score_weights)
    if global_score is not None:
        langfuse_client.create_score(
            trace_id=agg_trace_id,
            name="agg_score_global",
            value=float(global_score),
            data_type="NUMERIC",
            comment="Weighted global score across RAGAS means",
        )

    langfuse_client.create_event(
        trace_context={"trace_id": agg_trace_id},
        name="run_aggregate_metadata",
        input=run_metadata,
        output={"run_name": run_name, "rows": int(len(df_results)), "score_global": global_score},
    )


def _build_run_name(config: dict, retrieval_mode: str) -> str:
    alpha_value = config.get("hybrid_alpha") if bool(config.get("use_hybrid", False)) else "na"
    return (
        f"llm={config['llm']}|emb={config['embedding']}|prompt={config['prompt']}|"
        f"mode={retrieval_mode}|mq={int(bool(config.get('use_multiquery')))}|"
        f"rr={int(bool(config.get('use_reranker')))}|hyb={int(bool(config.get('use_hybrid')))}|"
        f"a={alpha_value}|k={config['k']}|T={config['temperature']}|h={_config_hash(config)}"
    )


def _build_trace_name(config: dict, retrieval_mode: str, item_id: str) -> str:
    alpha_value = config.get("hybrid_alpha") if bool(config.get("use_hybrid", False)) else "na"
    return (
        f"rag_item|llm={config['llm']}|emb={config['embedding']}|prompt={config['prompt']}|"
        f"mode={retrieval_mode}|mq={int(bool(config.get('use_multiquery')))}|"
        f"rr={int(bool(config.get('use_reranker')))}|hyb={int(bool(config.get('use_hybrid')))}|"
        f"a={alpha_value}|T={config['temperature']}|h={_config_hash(config)}|item={item_id}"
    )


def _append_summary_row(
    summary_store: list,
    run_name: str,
    retrieval_mode: str,
    config: dict,
    df_results: pd.DataFrame,
    score_weights: Optional[dict] = None,
) -> dict:
    row = {
        "run_name": run_name,
        "config_hash": _config_hash(config),
        "embedding": config.get("embedding"),
        "llm": config.get("llm"),
        "prompt": config.get("prompt"),
        "retrieval_mode": retrieval_mode,
        "use_hybrid": bool(config.get("use_hybrid", False)),
        "hybrid_alpha": config.get("hybrid_alpha") if bool(config.get("use_hybrid", False)) else None,
        "use_multiquery": bool(config.get("use_multiquery", False)),
        "use_reranker": bool(config.get("use_reranker", False)),
        "k": config.get("k"),
        "temperature": config.get("temperature"),
        "n_items": int(len(df_results)),
    }

    for metric in METRIC_NAMES:
        if metric in df_results.columns:
            non_null = df_results[metric].dropna()
            row[f"{metric}_mean"] = float(non_null.mean()) if not non_null.empty else None
        else:
            row[f"{metric}_mean"] = None

    row["score_global"] = _compute_global_score_from_means(row, score_weights)
    summary_store.append(row)
    return row


def _expand_experiment_grid(experiment_grid: dict) -> List[dict]:
    keys, values = zip(*experiment_grid.items())
    raw_configs = [dict(zip(keys, v)) for v in itertools.product(*values)]

    expanded = []
    for cfg in raw_configs:
        if not bool(cfg.get("use_hybrid", False)):
            cfg = dict(cfg)
            cfg["hybrid_alpha"] = None
            expanded.append(cfg)
            continue

        alpha = cfg.get("hybrid_alpha")
        if alpha is None:
            continue
        expanded.append(cfg)

    # Deduplicación estable por hash canónico
    seen = set()
    deduped = []
    for cfg in expanded:
        stable = json.dumps(cfg, sort_keys=True, ensure_ascii=False)
        if stable in seen:
            continue
        seen.add(stable)
        deduped.append(cfg)
    return deduped


def _build_combination_df_records(exp_result) -> list:
    records = []
    for item_result in exp_result.item_results:
        eval_map = {ev.name: ev.value for ev in item_result.evaluations}
        records.append({
            "faithfulness": eval_map.get("ragas_faithfulness"),
            "answer_correctness": eval_map.get("ragas_answer_correctness"),
            "context_recall": eval_map.get("ragas_context_recall"),
            "context_precision": eval_map.get("ragas_context_precision"),
            "trace_id": item_result.trace_id,
            "item_id": item_result.item.id,
        })
    return records


def run_single_experiment(config, dataset_obj, w_client, geo_data, summary_store=None, exec_mode: str = "full", score_weights: Optional[dict] = None):
    from langfuse.experiment import Evaluation

    rag_chain, retriever, retrieval_mode = build_rag_pipeline(**config, weaviate_client=w_client, geojson_data=geo_data)
    run_name = _build_run_name(config, retrieval_mode)
    run_metadata = _build_run_metadata(config, retrieval_mode, exec_mode)
    print(f"\nLote: {run_name}")

    if not hasattr(dataset_obj, "run_experiment"):
        raise ValueError("dataset_obj debe ser un dataset de Langfuse con método run_experiment")

    def task(*, item, **kwargs):
        trace_name = _build_trace_name(config, retrieval_mode, str(item.id))
        trace_id = langfuse_client.create_trace_id(seed=f"{run_name}-{item.id}")

        with langfuse_client.start_as_current_observation(
            trace_context={"trace_id": trace_id},
            name=trace_name,
            as_type="generation",
            input=item.input,
            model=config["llm"],
            metadata={
                "dataset_item_id": item.id,
                "run_name": run_name,
                **run_metadata,
            },
        ):
            ans = rag_chain.invoke(item.input, config={
                "callbacks": [langfuse_handler],
                "metadata": {
                    "dataset_item_id": item.id,
                    "run_name": run_name,
                    "langfuse_trace_id": trace_id,
                    **run_metadata,
                },
            })
            langfuse_client.update_current_generation(output=ans)

        w_docs = retriever.invoke(item.input)
        g_docs = query_geojson(item.input, geo_data, top_k=max(5, int(config.get("k", 5))))
        merged_contexts = [d.page_content for d in (w_docs[:10] + g_docs[:10])]

        try:
            langfuse_client.create_event(
                trace_context={"trace_id": trace_id},
                name="retrieval_breakdown",
                input={"question": item.input},
                output={
                    "weaviate_docs": len(w_docs),
                    "geojson_docs": len(g_docs),
                    "merged_contexts": len(merged_contexts),
                },
                metadata={
                    "retrieval_mode": retrieval_mode,
                    "use_hybrid": bool(config.get("use_hybrid", False)),
                    "hybrid_alpha": config.get("hybrid_alpha") if bool(config.get("use_hybrid", False)) else None,
                    "use_multiquery": bool(config.get("use_multiquery", False)),
                    "use_reranker": bool(config.get("use_reranker", False)),
                    "config_hash": _config_hash(config),
                },
            )
        except Exception as e:
            print(f"[WARN] No se pudo registrar retrieval_breakdown para trace {trace_id}: {e}")

        return {
            "answer": ans,
            "retrieved_contexts": merged_contexts,
        }

    def _mk_evaluator(metric_name: str):
        def evaluator(*, input, output, expected_output, metadata, **kwargs):
            answer = output.get("answer", "") if isinstance(output, dict) else str(output)
            contexts = output.get("retrieved_contexts", []) if isinstance(output, dict) else []
            value = _safe_ragas_single_eval(
                metric_name=metric_name,
                user_input=input,
                response=answer,
                reference=expected_output,
                retrieved_contexts=contexts,
            )
            return Evaluation(name=f"ragas_{metric_name}", value=value)
        return evaluator

    def run_avg_evaluator(*, item_results, **kwargs):
        evals = []
        for metric_name in METRIC_NAMES:
            values = []
            eval_name = f"ragas_{metric_name}"
            for item_result in item_results:
                for ev in item_result.evaluations:
                    if ev.name == eval_name and ev.value is not None:
                        values.append(float(ev.value))
            avg_val = float(sum(values) / len(values)) if values else None
            evals.append(Evaluation(name=f"avg_{eval_name}", value=avg_val))
        return evals

    exp_result = dataset_obj.run_experiment(
        name="RAG-IDEArq-experiment",
        run_name=run_name,
        description="Evaluacion RAG con metricas RAGAS",
        task=task,
        evaluators=[
            _mk_evaluator("faithfulness"),
            _mk_evaluator("answer_correctness"),
            _mk_evaluator("context_recall"),
            _mk_evaluator("context_precision"),
        ],
        run_evaluators=[run_avg_evaluator],
        max_concurrency=1,
        metadata=run_metadata,
    )

    records = _build_combination_df_records(exp_result)
    df_results = pd.DataFrame(records)

    # Garantizar métricas por item también en trace Langfuse.
    for row in records:
        trace_id = row.get("trace_id")
        if not trace_id:
            continue
        for metric_name in METRIC_NAMES:
            value = row.get(metric_name)
            if value is None or pd.isna(value):
                continue
            _log_metric_to_langfuse(trace_id=trace_id, observation_id=None, metric_name=metric_name, metric_value=float(value))

    if summary_store is not None:
        _append_summary_row(summary_store, run_name, retrieval_mode, config, df_results, score_weights=score_weights)

    _log_aggregates_to_langfuse(run_name, config, retrieval_mode, exec_mode, df_results, score_weights=score_weights)
    langfuse_client.flush()
    print(f"✅ Run '{run_name}' completado con {len(df_results)} items evaluados.")
    return df_results

In [27]:
def _safe_ragas_single_eval(metric_name: str, user_input: str, response: str, reference: str, retrieved_contexts: List[str]) -> Optional[float]:
    """Evalúa una métrica RAGAS de forma estricta.

    Si falla el juez o el parseo de RAGAS, devuelve None y lo registra como warning.
    No aplica métricas heurísticas sustitutas.
    """
    metric_obj = None
    if metric_name == "faithfulness":
        metric_obj = Faithfulness(llm=ragas_llm)
    elif metric_name == "answer_correctness":
        metric_obj = AnswerCorrectness(llm=ragas_llm, embeddings=ragas_emb)
    elif metric_name == "context_recall":
        metric_obj = ContextRecall(llm=ragas_llm)
    elif metric_name == "context_precision":
        metric_obj = ContextPrecision(llm=ragas_llm)
    else:
        return None

    max_attempts = 2
    for attempt in range(1, max_attempts + 1):
        try:
            single_ds = EvaluationDataset(samples=[
                SingleTurnSample(
                    user_input=user_input,
                    response=response if not _is_garbage_response(response) else _GARBAGE_PLACEHOLDER,
                    reference=reference,
                    retrieved_contexts=retrieved_contexts[:12],
                )
            ])
            single_res = ragas_evaluate(dataset=single_ds, metrics=[metric_obj], run_config=ragas_run_config)
            single_df = single_res.to_pandas()
            if metric_name in single_df.columns and pd.notnull(single_df.iloc[0][metric_name]):
                return float(single_df.iloc[0][metric_name])
        except Exception as e:
            print(f"[WARN] RAGAS {metric_name} fallo intento {attempt}/{max_attempts}: {e}")

    return None

# Validación rápida: dataset temporal pequeño con run_experiment para comprobar Run Items con valores
from datetime import datetime

tmp_dataset_name = f"RAG-IDEArq-smoke-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
langfuse_client.create_dataset(name=tmp_dataset_name, description="Smoke test temporal run_experiment")

src_dataset = langfuse_client.get_dataset("RAG-IDEArq-eval-v2")
for it in src_dataset.items[:2]:
    langfuse_client.create_dataset_item(
        dataset_name=tmp_dataset_name,
        input=it.input,
        expected_output=it.expected_output,
        metadata={"source": "tmp-smoke"},
    )

tmp_dataset = langfuse_client.get_dataset(tmp_dataset_name)

test_cfg = {
    "embedding": "e5-large-instruct",
    "llm": "mistral-small-24b-instruct",
    "prompt": "prompt_zero_shot",
    "temperature": 0.3,
    "k": 5,
    "use_multiquery": True,
    "use_reranker": True,
    "use_hybrid": False,
    "hybrid_alpha": 0.5,
    "populate_run_items": True,
}

mini_summary = []
w_client_tmp = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
try:
    mini_df = run_single_experiment(test_cfg, tmp_dataset, w_client_tmp, geo_data, summary_store=mini_summary)
finally:
    w_client_tmp.close()

print("[OK] mini_df shape:", None if mini_df is None else mini_df.shape)
print("[OK] temp dataset:", tmp_dataset_name)
print("[OK] sample metrics:", None if mini_df is None else mini_df[["faithfulness", "answer_correctness", "context_recall", "context_precision"]].head(2).to_dict("records"))

### Ejecución final

Ejecuta una configuración RAG completa sobre el dataset, calcula métricas RAGAS, y registra todo como un experimento en Langfuse.

if __name__ == "__main__":
    # 1. Conexiones
    w_client = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
    geo_data = load_geojson(Path("capas/c14_v2.geojson"))

    # 2. CONFIGURACIÓN DEL EXPERIMENTO
    config = {
        "embedding": "e5-large-instruct",
        "llm": "qwen2.5:7b",
        "prompt": "prompt_zero_shot",
        "temperature": 0.3,
        "k": 5
    }
    DATASET_NAME = "RAG-IDEArq-eval-v2"
    # 3. BAJAR EL DATASET DIRECTAMENTE DESDE LANGFUSE
    print(f"Recuperando dataset '{DATASET_NAME}' desde Langfuse...")
    dataset = langfuse_client.get_dataset(DATASET_NAME)

    # 4. REFORMATEAR PARA TU FUNCIÓN (Mapeamos input -> question y expected_output -> ground_truth)
    # Esto adapta lo que viene de Langfuse a lo que espera tu función run_single_experiment
    eval_examples_from_lf = [
        {
            "inputs": {"question": item.input}, 
            "outputs": {"ground_truth": item.expected_output}
        } 
        for item in dataset.items
    ]

    print(f"Se han recuperado {len(eval_examples_from_lf)} casos de prueba.")

    # 5. LANZAR EXPERIMENTO
    run_single_experiment(config, eval_examples_from_lf, w_client, geo_data)

    langfuse_client.flush()
    print("\nProceso terminado. Todos los datos enviados a Langfuse.")

In [22]:
import weaviate

client = weaviate.connect_to_local(
    host="localhost",
    port=8080,
    grpc_port=50051
)
print(client.is_ready())

True


/home/raglinux/env_rag/lib/python3.12/site-packages/weaviate/warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
/tmp/ipykernel_2649507/3273426004.py:3: ResourceWarning: unclosed <socket.socket fd=89, family=2, type=1, proto=6, laddr=('127.0.0.1', 40772), raddr=('127.0.0.1', 8080)>
  client = weaviate.connect_to_local(


In [ ]:
if __name__ == "__main__":
    # Grid de experimentos (sin re-ingesta, solo evaluación sobre Weaviate ya indexado)
    # Dimensiones obligatorias solicitadas: prompt, embedding, generador, multiquery, reranker, híbrida, temperatura.
    experiment_grid = {
        "embedding": ["e5-large-instruct"],
        "llm": ["mistral-small-24b-instruct"],
        "prompt": ["prompt_zero_shot", "prompt_one_shot", "prompt_few_shot"],
        "temperature": [0.3, 0.5, 0.7],
        "k": [10],
        "k_multi": [20],
        "reranker_top_n": [5],
        "use_multiquery": [True, False],
        "use_reranker": [True, False],
        "use_hybrid": [False, True],
        "hybrid_alpha": [0.3, 0.5, 0.7],
        "populate_run_items": [True],
    }

    # Pesos del score global compuesto (normalizados internamente).
    score_weights = {
        "faithfulness": 0.25,
        "answer_correctness": 0.25,
        "context_recall": 0.25,
        "context_precision": 0.25,
    }

    configs = _expand_experiment_grid(experiment_grid)

    # Ejecutar barrido completo de combinaciones.
    SMOKE_TEST = False
    SMOKE_ITEMS = 3

    EXPERIMENT_DATASET_NAME = "RAG-IDEArq-eval-v2"
    ACTIVE_COLLECTION = "BgeE5Chunk1000Sim05"

    w_client = None
    try:
        w_client = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)

        # Garantiza que la colección de Weaviate coincide con la solicitada.
        emb_collection = EMBEDDING_CONFIGS["e5-large-instruct"]["weaviate_index"]
        if emb_collection != ACTIVE_COLLECTION:
            raise RuntimeError(
                f"Desalineación de colección: EMBEDDING_CONFIGS usa '{emb_collection}' y se requiere '{ACTIVE_COLLECTION}'."
            )

        _preflight_or_raise(
            w_client,
            collection_name=ACTIVE_COLLECTION,
            dataset_name=EXPERIMENT_DATASET_NAME,
            generator_model="mistral-small-24b-instruct",
        )

        geo_data = load_geojson(GEOJSON_PATH)
        dataset = langfuse_client.get_dataset(EXPERIMENT_DATASET_NAME)
        dataset_source = dataset

        # Configuración de juez local (offline): modelo válido en Ollama.
        ragas_judge_model = os.getenv("RAGAS_JUDGE_MODEL", "qwen2.5:14b")
        raw_judge = ChatOllama(model=ragas_judge_model, temperature=0.1, format="json", num_ctx=32768)
        ragas_llm = JSONCleaningLLM(raw_judge)
        ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))

        ragas_metrics = [
            Faithfulness(llm=ragas_llm),
            AnswerCorrectness(llm=ragas_llm, embeddings=ragas_emb),
            ContextRecall(llm=ragas_llm),
            ContextPrecision(llm=ragas_llm),
        ]
        ragas_run_config = RunConfig(max_workers=1, timeout=1200)

        summary_rows = []
        total_configs = len(configs)

        if SMOKE_TEST:
            configs = configs[:1]
            tmp_dataset_name = f"RAG-IDEArq-smoke-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
            langfuse_client.create_dataset(name=tmp_dataset_name, description="Smoke test temporal run_experiment")
            for it in dataset.items[:SMOKE_ITEMS]:
                langfuse_client.create_dataset_item(
                    dataset_name=tmp_dataset_name,
                    input=it.input,
                    expected_output=it.expected_output,
                    metadata={"source": "tmp-smoke"},
                )
            dataset_source = langfuse_client.get_dataset(tmp_dataset_name)
            exec_mode = "smoke"
            print(f"[INFO] SMOKE_TEST activo: {SMOKE_ITEMS} items y {len(configs)} configuración de {total_configs} totales.")
        else:
            exec_mode = "full"
            print(f"[INFO] Modo completo: {len(dataset.items)} items y {len(configs)} configuraciones (esperadas: 144).")

        for i, cfg in enumerate(configs):
            print(f"\n--- CONFIG {i + 1}/{len(configs)} ---")
            try:
                run_single_experiment(
                    cfg,
                    dataset_source,
                    w_client,
                    geo_data,
                    summary_store=summary_rows,
                    exec_mode=exec_mode,
                    score_weights=score_weights,
                )
            except Exception as e:
                print(f"❌ Fallo en experimento: {e}")
                continue

        if summary_rows:
            summary_df = pd.DataFrame(summary_rows)
            metric_cols = [
                "faithfulness_mean",
                "answer_correctness_mean",
                "context_recall_mean",
                "context_precision_mean",
                "score_global",
            ]
            show_cols = [
                "run_name",
                "config_hash",
                "llm",
                "embedding",
                "prompt",
                "retrieval_mode",
                "use_hybrid",
                "hybrid_alpha",
                "use_multiquery",
                "use_reranker",
                "k",
                "k_multi",
                "reranker_top_n",
                "temperature",
                "n_items",
            ] + metric_cols

            print("\n===== PROMEDIOS POR COMBINACIÓN =====")
            ranking_global = summary_df[show_cols].sort_values(by=["score_global", "answer_correctness_mean"], ascending=False)
            display(ranking_global)

            print("\n===== TOP 10 POR SCORE GLOBAL =====")
            display(ranking_global.head(10))

            for metric in ["faithfulness_mean", "answer_correctness_mean", "context_recall_mean", "context_precision_mean"]:
                print(f"\n===== TOP 10 POR {metric.upper()} =====")
                metric_rank = summary_df[show_cols].sort_values(by=[metric, "score_global"], ascending=False)
                display(metric_rank.head(10))

            out_csv = Path("results/langfuse_ragas_summary.csv")
            out_csv.parent.mkdir(parents=True, exist_ok=True)
            summary_df.to_csv(out_csv, index=False)
            print(f"[INFO] Resumen exportado a: {out_csv}")

        langfuse_client.flush()
        print("\n🏁 PROCESO FINALIZADO. Revisa Run Items, trazas y métricas ragas_* por item, agregados agg_* y agg_score_global en Langfuse.")
    finally:
        if w_client is not None:
            w_client.close()
            print("[INFO] Conexión Weaviate cerrada.")

/home/raglinux/env_rag/lib/python3.12/site-packages/shapely/io.py:353: ResourceWarning: unclosed <socket.socket fd=103, family=2, type=1, proto=6, laddr=('127.0.0.1', 39428), raddr=('127.0.0.1', 11434)>
  return lib.from_wkb(geometry, invalid_handler, **kwargs)


[INFO] CRS detectado: EPSG:4258. Reproyectando a EPSG:4326.


/tmp/ipykernel_1310907/3954033599.py:61: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; client = OpenAI(api_key='...'); llm = llm_factory('gpt-4o-mini', client=client)
  ragas_llm = JSONCleaningLLM(raw_judge)
/tmp/ipykernel_1310907/3954033599.py:62: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
/home/raglinux/env_rag/lib/python3.12/site-packages/ragas/utils.py:113: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use t

[INFO] Modo completo: 51 items y 144 configuraciones (esperadas: 144).

--- CONFIG 1/144 ---

Lote: llm=mistral-small-24b-instruct|emb=e5-large-instruct|prompt=prompt_zero_shot|mode=similarity|mq=1|rr=1|hyb=0|a=na|k=10|T=0.3|h=1185a070e5bc


Propagated attribute 'experiment_metadata' value is over 200 characters (536 chars). Dropping value.
Propagated attribute 'experiment_metadata' value is over 200 characters (536 chars). Dropping value.
/home/raglinux/env_rag/lib/python3.12/site-packages/ragas/_analytics.py:278: DeprecationWarning: evaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result = func(*args, **kwargs)
/home/raglinux/env_rag/lib/python3.12/site-packages/ragas/evaluation.py:457: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  return await aevaluate(


...

[INFO] Conexión Weaviate cerrada.


KeyboardInterrupt: 

Propagated attribute 'experiment_metadata' value is over 200 characters (536 chars). Dropping value.
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/ipykernel/iostream.py", line 694, in write
    self._schedule_flush()
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/ipykernel/iostream.py", line 590, in _schedule_flush
    self.pub_thread.schedule(_schedule_in_thread)
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/ipykernel/iostream.py", line 267, in schedule
    self._event_pipe.send(b"")
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/zmq/sugar/socket.py", line 698, in send
    return super().send(data, flags=flags, copy=copy, track=track)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "zmq/backend/cython/_zmq.py", line 1152, in zmq.backend.cython

[WARN] RAGAS answer_correctness fallo intento 1/2: Socket operation on non-socket[WARN] RAGAS context_recall fallo intento 1/2: Socket operation on non-socket
[WARN] RAGAS context_recall fallo intento 2/2: Socket operation on non-socket
[WARN] RAGAS context_precision fallo intento 1/2: Socket operation on non-socket
[WARN] RAGAS context_precision fallo intento 2/2: Socket operation on non-socket


Propagated attribute 'experiment_metadata' value is over 200 characters (536 chars). Dropping value.
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/ipykernel/iostream.py", line 694, in write
    self._schedule_flush()
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/ipykernel/iostream.py", line 590, in _schedule_flush
    self.pub_thread.schedule(_schedule_in_thread)
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/ipykernel/iostream.py", line 267, in schedule
    self._event_pipe.send(b"")
  File "/home/raglinux/env_rag/lib/python3.12/site-packages/zmq/sugar/socket.py", line 698, in send
    return super().send(data, flags=flags, copy=copy, track=track)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "zmq/backend/cython/_zmq.py", line 1152, in zmq.backend.cython

In [ ]:
# Comprobación directa de la colección activa
w_client_check = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
try:
    collection_name = EMBEDDING_CONFIGS["e5-large-instruct"]["weaviate_index"]
    res = w_client_check.collections.get(collection_name).query.fetch_objects(limit=1)
    print(f"[OK] Colección '{collection_name}' accesible. Objetos recuperados: {len(res.objects)}")
    if res.objects:
        print(res.objects[0].properties)
finally:
    w_client_check.close()

{'title': '', 'file_type': 'pdf', 'total_chunks': 2954.0, 'filename': '621_2006.pdf', 'text': None, 'total_pages': 352.0, 'producer': None, 'is_table': None, 'creationDate': None, 'page': None, 'chunk_size': 233.0, 'creator': None, 'chunk_index': 2009.0, 'file_path': None, 'chunking_method': 'semantic_chonkie', 'loader': None, 'doc_length': 826317.0, 'content': 'Servía de tapadera a la urna 4-2. Diám. boca: 12.8 cm. Diám. base: 3.8 cm. H.: 5 cm. 4-4. Urna gris de tipo D1B, con borde exvasado, cue- llo recto, cuerpo globular, sin pie y base plana. A tor- no. Fuego reductor. Pasta gris clara. ', 'source': 'ingesta/621_2006.pdf'}


### Ejecutar todos los experimentos (grid de configuraciones)

Genera la combinatoria de configuraciones y ejecuta cada una como un experimento independiente.

Faithfulness: ¿La respuesta es fiel al contexto?

Answer Correctness: ¿La respuesta es correcta comparada con el ground truth?

Context Recall: ¿El retriever encontró lo que el ground truth decía?

Context Precision: ¿Los documentos útiles estaban arriba en la lista?

In [33]:
# Smoke test de validación end-to-end (1 config, 3 items)
from datetime import datetime

smoke_items = 3
tmp_dataset_name = f"RAG-IDEArq-smoke-rerun-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
langfuse_client.create_dataset(name=tmp_dataset_name, description="Smoke test temporal rerun")

base_dataset = langfuse_client.get_dataset("RAG-IDEArq-eval-v2")
for it in base_dataset.items[:smoke_items]:
    langfuse_client.create_dataset_item(
        dataset_name=tmp_dataset_name,
        input=it.input,
        expected_output=it.expected_output,
        metadata={"source": "tmp-smoke-rerun"},
    )

smoke_dataset = langfuse_client.get_dataset(tmp_dataset_name)

smoke_cfg = {
    "embedding": "e5-large-instruct",
    "llm": "mistral-small-24b-instruct",
    "prompt": "prompt_zero_shot",
    "temperature": 0.3,
    "k": 10,
    "k_multi": 20,
    "reranker_top_n": 5,
    "use_multiquery": True,
    "use_reranker": True,
    "use_hybrid": True,
    "hybrid_alpha": 0.5,
    "populate_run_items": True,
}

mini_summary = []
w_client_smoke = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
try:
    geo_data_smoke = load_geojson(GEOJSON_PATH)
    mini_df = run_single_experiment(
        smoke_cfg,
        smoke_dataset,
        w_client_smoke,
        geo_data_smoke,
        summary_store=mini_summary,
        exec_mode="smoke",
        score_weights=DEFAULT_SCORE_WEIGHTS,
    )
finally:
    w_client_smoke.close()

print(f"[OK] Smoke dataset: {tmp_dataset_name}")
print("[OK] mini_df shape:", mini_df.shape if mini_df is not None else None)
if mini_df is not None and not mini_df.empty:
    display(mini_df[["item_id", "faithfulness", "answer_correctness", "context_recall", "context_precision"]].head())
    print("[OK] Medias:", mini_df[["faithfulness", "answer_correctness", "context_recall", "context_precision"]].mean(numeric_only=True).to_dict())

[INFO] CRS detectado: EPSG:4258. Reproyectando a EPSG:4326.


Propagated attribute 'experiment_metadata' value is over 200 characters (538 chars). Dropping value.



Lote: llm=mistral-small-24b-instruct|emb=e5-large-instruct|prompt=prompt_zero_shot|mode=hybrid_native|mq=1|rr=1|hyb=1|a=0.5|k=10|T=0.3|h=120b9510d827


Propagated attribute 'experiment_metadata' value is over 200 characters (538 chars). Dropping value.
/home/raglinux/env_rag/lib/python3.12/site-packages/ragas/_analytics.py:278: DeprecationWarning: evaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result = func(*args, **kwargs)
/home/raglinux/env_rag/lib/python3.12/site-packages/ragas/evaluation.py:457: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  return await aevaluate(


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

...

✅ Run 'llm=mistral-small-24b-instruct|emb=e5-large-instruct|prompt=prompt_zero_shot|mode=hybrid_native|mq=1|rr=1|hyb=1|a=0.5|k=10|T=0.3|h=120b9510d827' completado con 3 items evaluados.


/tmp/ipykernel_1310907/3640636259.py:38: ResourceWarning: unclosed <socket.socket fd=112, family=2, type=1, proto=6, laddr=('127.0.0.1', 43556), raddr=('127.0.0.1', 11434)>
  mini_df = run_single_experiment(
/tmp/ipykernel_1310907/3640636259.py:38: ResourceWarning: unclosed <socket.socket fd=111, family=2, type=1, proto=6, laddr=('127.0.0.1', 34280), raddr=('127.0.0.1', 11434)>
  mini_df = run_single_experiment(


[OK] Smoke dataset: RAG-IDEArq-smoke-rerun-20260422-133440
[OK] mini_df shape: (3, 6)


,item_id,faithfulness,answer_correctness,context_recall,context_precision
0,2ed3f044-f008-46a2-9e5a-fd372d651f48,0.700000,0.269917,0.0,0.000000
1,b839ff08-1dd3-49a0-8bee-5a305e07a6de,0.230769,0.640348,0.0,0.250000
2,d26c4b1e-54f2-4329-8f59-66c3d36f3b27,0.090909,0.450977,0.0,0.083333


[OK] Medias: {'faithfulness': 0.34055944055944054, 'answer_correctness': 0.4537474864615665, 'context_recall': 0.0, 'context_precision': 0.11111111109999999}


In [23]:
# Smoke test rerun (similarity only) para validar pipeline completo
from datetime import datetime

smoke_items = 3
tmp_dataset_name_2 = f"RAG-IDEArq-smoke-rerun-sim-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
langfuse_client.create_dataset(name=tmp_dataset_name_2, description="Smoke test temporal rerun similarity")

base_dataset_2 = langfuse_client.get_dataset("RAG-IDEArq-eval-v2")
for it in base_dataset_2.items[:smoke_items]:
    langfuse_client.create_dataset_item(
        dataset_name=tmp_dataset_name_2,
        input=it.input,
        expected_output=it.expected_output,
        metadata={"source": "tmp-smoke-rerun-sim"},
    )

smoke_dataset_2 = langfuse_client.get_dataset(tmp_dataset_name_2)

smoke_cfg_2 = {
    "embedding": "e5-large-instruct",
    "llm": "mistral-small-24b-instruct",
    "prompt": "prompt_zero_shot",
    "temperature": 0.3,
    "k": 10,
    "k_multi": 20,
    "reranker_top_n": 5,
    "use_multiquery": True,
    "use_reranker": True,
    "use_hybrid": False,
    "hybrid_alpha": None,
    "populate_run_items": True,
}

mini_summary_2 = []
w_client_smoke_2 = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
try:
    geo_data_smoke_2 = load_geojson(GEOJSON_PATH)
    mini_df_2 = run_single_experiment(
        smoke_cfg_2,
        smoke_dataset_2,
        w_client_smoke_2,
        geo_data_smoke_2,
        summary_store=mini_summary_2,
        exec_mode="smoke",
        score_weights=DEFAULT_SCORE_WEIGHTS,
    )
finally:
    w_client_smoke_2.close()

print(f"[OK] Smoke dataset: {tmp_dataset_name_2}")
print("[OK] mini_df shape:", mini_df_2.shape if mini_df_2 is not None else None)
if mini_df_2 is not None and not mini_df_2.empty:
    display(mini_df_2[["item_id", "faithfulness", "answer_correctness", "context_recall", "context_precision"]].head())
    print("[OK] Medias:", mini_df_2[["faithfulness", "answer_correctness", "context_recall", "context_precision"]].mean(numeric_only=True).to_dict())

[INFO] CRS detectado: EPSG:4258. Reproyectando a EPSG:4326.


Propagated attribute 'experiment_metadata' value is over 200 characters (537 chars). Dropping value.



Lote: llm=mistral-small-24b-instruct|emb=e5-large-instruct|prompt=prompt_zero_shot|mode=similarity|mq=1|rr=1|hyb=0|a=na|k=10|T=0.3|h=1185a070e5bc


Propagated attribute 'experiment_metadata' value is over 200 characters (537 chars). Dropping value.
/home/raglinux/env_rag/lib/python3.12/site-packages/ragas/_analytics.py:278: DeprecationWarning: evaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result = func(*args, **kwargs)
/home/raglinux/env_rag/lib/python3.12/site-packages/ragas/evaluation.py:457: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  return await aevaluate(


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Run 'llm=mistral-small-24b-instruct|emb=e5-large-instruct|prompt=prompt_zero_shot|mode=similarity|mq=1|rr=1|hyb=0|a=na|k=10|T=0.3|h=1185a070e5bc' completado con 3 items evaluados.
[OK] Smoke dataset: RAG-IDEArq-smoke-rerun-sim-20260422-125938
[OK] mini_df shape: (3, 6)


/tmp/ipykernel_1310907/4102289626.py:38: ResourceWarning: unclosed <socket.socket fd=102, family=2, type=1, proto=6, laddr=('127.0.0.1', 47802), raddr=('127.0.0.1', 11434)>
  mini_df_2 = run_single_experiment(
/tmp/ipykernel_1310907/4102289626.py:38: ResourceWarning: unclosed <socket.socket fd=101, family=2, type=1, proto=6, laddr=('127.0.0.1', 53264), raddr=('127.0.0.1', 11434)>
  mini_df_2 = run_single_experiment(


,item_id,faithfulness,answer_correctness,context_recall,context_precision
0,b0c9e042-1e57-4108-ba9a-e1597d566f9c,0.75,0.105750,0.0,0.0
1,19999091-d969-444a-9963-0673cf8e9bf0,0.00,0.654540,0.0,1.0
2,b0a6aae0-39ec-405a-bf91-7497e57fc5f2,0.00,0.462403,0.0,0.0


[OK] Medias: {'faithfulness': 0.25, 'answer_correctness': 0.40756432034784024, 'context_recall': 0.0, 'context_precision': 0.3333333333}


In [31]:
# Sanity check: híbrida nativa de Weaviate en una consulta corta
quick_cfg = {
    "embedding": "e5-large-instruct",
    "llm": "mistral-small-24b-instruct",
    "prompt": "prompt_zero_shot",
    "temperature": 0.3,
    "k": 10,
    "k_multi": 20,
    "reranker_top_n": 5,
    "use_multiquery": False,
    "use_reranker": False,
    "use_hybrid": True,
    "hybrid_alpha": 0.5,
}

w_quick = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
try:
    geo_quick = load_geojson(GEOJSON_PATH)
    _, quick_retriever, quick_mode = build_rag_pipeline(
        **quick_cfg,
        weaviate_client=w_quick,
        geojson_data=geo_quick,
    )
    quick_docs = quick_retriever.invoke("Yacimientos neolíticos situados a menos de 150 km de Casa Montero")
    print("[OK] retrieval_mode:", quick_mode)
    print("[OK] docs recuperados:", len(quick_docs))
    if quick_docs:
        print("[OK] metadata primer doc:", quick_docs[0].metadata)
finally:
    w_quick.close()

[INFO] CRS detectado: EPSG:4258. Reproyectando a EPSG:4326.
[OK] retrieval_mode: hybrid_native
[OK] docs recuperados: 10
[OK] metadata primer doc: {'source': 'weaviate_hybrid', 'uuid': '9d914b0f-cd01-491a-b774-76e6cb44d121'}


In [35]:
# Check rápido de cálculo de distancia en km (Haversine)
geo_check = load_geojson(GEOJSON_PATH)
km_docs = query_geojson("Yacimientos neolíticos situados a menos de 150 km de Casa Montero", geo_check, top_k=3)
print("[OK] docs geo:", len(km_docs))
if km_docs:
    print(km_docs[0].page_content.split("\n")[0])
    print(km_docs[0].metadata)

[INFO] CRS detectado: EPSG:4258. Reproyectando a EPSG:4326.


/usr/lib/python3.12/asyncio/selector_events.py:879: ResourceWarning: unclosed transport <_SelectorSocketTransport closing fd=103 read=idle write=<idle, bufsize=0>>
  _warn(f"unclosed transport {self!r}", ResourceWarning, source=self)
/usr/lib/python3.12/asyncio/base_events.py:730: ResourceWarning: unclosed event loop <_UnixSelectorEventLoop running=False closed=False debug=False>
  _warn(f"unclosed event loop {self!r}", ResourceWarning, source=self)


[OK] docs geo: 3
DISTANCIA_GEO: Distancia entre 'Casa Montero' y 'Casa Montero': 0.00 km (Euclidiana).
{'source': 'capas/c14_v2.geojson', 'feature_index': '1228', 'distance_km': 0.0, 'anchor_site': 'Casa Montero', 'yacimiento': 'Casa Montero'}


In [ ]:
# Crear dataset derivado con hechos mínimos en metadata (sin modificar el dataset original)
from datetime import datetime

SOURCE_DATASET_NAME = "RAG-IDEArq-eval-v2"
TARGET_DATASET_NAME = f"RAG-IDEArq-eval-v2-minfacts-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Hechos mínimos por pregunta exacta
MIN_FACTS_BY_QUESTION = {
    "What are the main theoretical models of Neolithic expansion in Europe?": [
        "demic diffusion",
        "cultural diffusion",
    ],
    "Quais são as datas mais antigas da extração de sílex na península central?": [
        "Casa Montero",
        "5327-5215 cal BC",
    ],
    "¿Cuáles son las cronologías de las manifestaciones funerarias del Mesolítico en las distintas regiones peninsulares?": [
        "Mediterráneo más temprano",
        "Portugal atlántico intermedio",
        "costa cantábrica más tardía",
    ],
    "Principales yacimientos de la Segunda Edad del Hierro en la provincia de León.": [
        "Lancia",
        "Chano",
        "Peña del Castro",
    ],
    "Periodización del Bronce Final en el Levante de la Península Ibérica, cronología de las fases y principales ejemplos de yacimientos asignados a las mismas.": [
        "Bronce tardío o reciente",
        "Bronce final I",
        "Bronce final II",
        "Bronce final III",
    ],
    "Yacimientos Calcolíticos de la Península Ibérica  en los que se han hallado objetos de marfil.": [
        "Valencina de la Concepción",
        "Los Millares",
        "Perdigões",
    ],
    "Cronología y districubión espacial del poblamiento neolítico en la Meseta Sur.": [
        "valle medio y alto del Tajo",
        "La Mancha",
        "distribución desigual",
    ],
    "Excavaciones de urgencia de la Junta de Andalucía en la provincia de Almería publicadas en 2001.": [
        "av. Pablo Iglesias esquina Rafaela Jiménez",
        "calle La Central de Villaricos",
        "calle Castillejo (Gádor)",
    ],
    "In what year did the Siret brothers excavate the La Bastida de Totana site?": [
        "1886",
    ],
    "Yacimiento com a data de C14 mais antiga das Ilhas Baleares.": [
        "Cova de Moleta (Sóller)",
    ],
    "Yacimientos neolíticos situados a menos de 150 km de Casa Montero": [
        "El Cañaveral",
        "Pista de Motos",
        "O'Donnell",
        "Cueva de la Higuera",
    ],
    "Datación más antigua y más reciente de los yacimientos calcolíticos en el área de Valencina de la concepción (Sevilla)": [
        "más antigua: 4800 +- 100",
        "más reciente: 175 +- 20",
    ],
    "Dime si esta datación es la más reciente de los yacimientos calcolíticos en el área de Valencia de la concepción (Sevilla): Valencina, Cerro de la Cabeza, Ladera Sur, -1377 +- 23 ": [
        "respuesta: no",
        "datación correcta más reciente: 175 +- 20",
    ],
    "Fecha más antigua para un yacimiento funerario megalítico en la Península Ibérica.": [
        "Trikuaizti 2",
        "12015 +- 145",
    ],
    "Dataciones más antiguas (i.e, más altas) de yacimientos paleolíticos para cada comunidad autónoma": [
        "respuesta en formato lista por comunidad",
    ],
    "¿Estas dataciones son del Neolítico? Comunitat Valenciana, 33.900 ± 60, Galicia, 31.690 ± 50, Región de Murcia, 12.030 ± 0": [
        "respuesta: no",
        "corresponden al Paleolítico",
    ],
    "¿Cuál es el yacimiento calcolítico más alejado de la ciudad de Jaén en la provincia de Jaén?": [
        "Eras del Alcázar",
    ],
}

src = langfuse_client.get_dataset(SOURCE_DATASET_NAME)
langfuse_client.create_dataset(
    name=TARGET_DATASET_NAME,
    description="Dataset derivado con hechos_minimos para evaluación RAG",
)

created = 0
missing_map = 0
for item in src.items:
    q = str(item.input)
    facts = MIN_FACTS_BY_QUESTION.get(q, [])
    if not facts:
        missing_map += 1

    base_meta = dict(item.metadata or {})
    base_meta.update({
        "hechos_minimos": facts,
        "n_hechos_minimos": len(facts),
        "eval_schema": "minfacts_v1",
    })

    langfuse_client.create_dataset_item(
        dataset_name=TARGET_DATASET_NAME,
        input=item.input,
        expected_output=item.expected_output,
        metadata=base_meta,
    )
    created += 1

print(f"[OK] Dataset origen: {SOURCE_DATASET_NAME}")
print(f"[OK] Dataset destino: {TARGET_DATASET_NAME}")
print(f"[OK] Items copiados: {created}")
print(f"[INFO] Preguntas sin mapeo de hechos mínimos: {missing_map}")